# Data Collection & Cleaning

Combines four quarterly BTS DB1B Market files (Q3 2024 – Q2 2025), removes incomplete records, and joins airport lat/lon from OurAirports for use as GNN node features.

In [2]:
import pandas as pd
import numpy as np
import os

## 1. Preview Raw Data

In [3]:
sample = pd.read_csv('../../data/rawdata/2024_q3.csv', low_memory=False)
print(f"Shape: {sample.shape}")
print(f"Columns: {list(sample.columns)}")
sample.head()

Shape: (8297869, 15)
Columns: ['YEAR', 'QUARTER', 'ORIGIN_AIRPORT_ID', 'ORIGIN', 'DEST_AIRPORT_ID', 'DEST', 'REPORTING_CARRIER', 'TICKET_CARRIER', 'OPERATING_CARRIER', 'BULK_FARE', 'PASSENGERS', 'MARKET_FARE', 'MARKET_DISTANCE', 'NONSTOP_MILES', 'MKT_GEO_TYPE']


,YEAR,QUARTER,ORIGIN_AIRPORT_ID,ORIGIN,DEST_AIRPORT_ID,DEST,REPORTING_CARRIER,TICKET_CARRIER,OPERATING_CARRIER,BULK_FARE,PASSENGERS,MARKET_FARE,MARKET_DISTANCE,NONSTOP_MILES,MKT_GEO_TYPE
0,2024,3,10135,ABE,10136,ABI,MQ,AA,99,0.0,1.0,430.5,1575.0,1458.0,2
1,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,5.5,1961.0,1738.0,2
2,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,148.5,1961.0,1738.0,2
3,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,182.0,1961.0,1738.0,2
4,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,309.5,1961.0,1738.0,2


## 2. Combine Quarterly Files

In [4]:
files = [
    '../../data/rawdata/2024_q3.csv',
    '../../data/rawdata/2024_q4.csv',
    '../../data/rawdata/2025_q1.csv',
    '../../data/rawdata/2025_q2.csv',
]

dfs = []
for path in files:
    df = pd.read_csv(path, low_memory=False)
    print(f"  {os.path.basename(path)}: {df.shape}")
    dfs.append(df)

combined = pd.concat(dfs, ignore_index=True)
print(f"\nCombined shape: {combined.shape}")
print(f"Quarters present: {sorted(combined[['YEAR','QUARTER']].drop_duplicates().apply(tuple, axis=1).tolist())}")
combined.head()

  2024_q3.csv: (8297869, 15)
  2024_q4.csv: (8525077, 15)
  2025_q1.csv: (7297028, 15)
  2025_q2.csv: (8450420, 15)

Combined shape: (32570394, 15)
Quarters present: [(2024, 3), (2024, 4), (2025, 1), (2025, 2)]


,YEAR,QUARTER,ORIGIN_AIRPORT_ID,ORIGIN,DEST_AIRPORT_ID,DEST,REPORTING_CARRIER,TICKET_CARRIER,OPERATING_CARRIER,BULK_FARE,PASSENGERS,MARKET_FARE,MARKET_DISTANCE,NONSTOP_MILES,MKT_GEO_TYPE
0,2024,3,10135,ABE,10136,ABI,MQ,AA,99,0.0,1.0,430.5,1575.0,1458.0,2
1,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,5.5,1961.0,1738.0,2
2,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,148.5,1961.0,1738.0,2
3,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,182.0,1961.0,1738.0,2
4,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,309.5,1961.0,1738.0,2


## 3. Missing Value Analysis

In [5]:
print("=== Missing Value Counts ===")
print(combined.isna().sum())
print(f"\nRows with any missing value: {combined.isna().any(axis=1).sum():,}  ({combined.isna().any(axis=1).mean():.1%} of total)")

=== Missing Value Counts ===
YEAR                 0
QUARTER              0
ORIGIN_AIRPORT_ID    0
ORIGIN               0
DEST_AIRPORT_ID      0
DEST                 0
REPORTING_CARRIER    0
TICKET_CARRIER       0
OPERATING_CARRIER    0
BULK_FARE            0
PASSENGERS           0
MARKET_FARE          0
MARKET_DISTANCE      0
NONSTOP_MILES        0
MKT_GEO_TYPE         0
dtype: int64

Rows with any missing value: 0  (0.0% of total)


## 4. Clean Data

Drop rows with missing values and remove zero/negative fares (data entry artifacts).

In [6]:
df_clean = (
    combined
    .dropna()
    .query("MARKET_FARE > 0")
    .reset_index(drop=True)
)

print(f"Before cleaning: {combined.shape[0]:,} rows")
print(f"After cleaning:  {df_clean.shape[0]:,} rows")
print(f"Removed:         {combined.shape[0] - df_clean.shape[0]:,} rows ({(combined.shape[0] - df_clean.shape[0]) / combined.shape[0]:.1%})")
df_clean.describe()

Before cleaning: 32,570,394 rows
After cleaning:  32,541,848 rows
Removed:         28,546 rows (0.1%)


,YEAR,QUARTER,ORIGIN_AIRPORT_ID,DEST_AIRPORT_ID,BULK_FARE,PASSENGERS,MARKET_FARE,MARKET_DISTANCE,NONSTOP_MILES,MKT_GEO_TYPE
count,3.254185e+07,3.254185e+07,3.254185e+07,3.254185e+07,32541848.0,3.254185e+07,3.254185e+07,3.254185e+07,3.254185e+07,3.254185e+07
mean,2.024483e+03,2.554236e+00,1.282923e+04,1.282866e+04,0.0,1.895133e+00,2.710958e+02,1.301878e+03,1.221670e+03,1.932522e+00
std,4.997272e-01,1.103920e+00,1.541862e+03,1.542257e+03,0.0,4.971489e+00,2.216202e+02,8.166206e+02,7.736060e+02,2.508474e-01
min,2.024000e+03,1.000000e+00,1.013500e+04,1.013500e+04,0.0,1.000000e+00,1.100000e-01,1.100000e+01,1.100000e+01,1.000000e+00
25%,2.024000e+03,2.000000e+00,1.129800e+04,1.129800e+04,0.0,1.000000e+00,1.490000e+02,7.100000e+02,6.540000e+02,2.000000e+00
50%,2.024000e+03,3.000000e+00,1.294500e+04,1.294500e+04,0.0,1.000000e+00,2.310000e+02,1.087000e+03,1.020000e+03,2.000000e+00
75%,2.025000e+03,4.000000e+00,1.410700e+04,1.410700e+04,0.0,1.000000e+00,3.400000e+02,1.751000e+03,1.643000e+03,2.000000e+00
max,2.025000e+03,4.000000e+00,1.686900e+04,1.686900e+04,0.0,1.032000e+03,4.443200e+04,1.134400e+04,9.411000e+03,2.000000e+00


## 5. Join Airport Coordinates

`airports.dat.txt` (OurAirports) provides latitude and longitude keyed on IATA code. We join on `ORIGIN` and `DEST` to add geo features needed by the GNN node embeddings.

In [7]:
airport_cols = ['airport_id', 'name', 'city', 'country', 'iata', 'icao',
                'latitude', 'longitude', 'altitude', 'timezone', 'dst', 'tz_db', 'type', 'source']

airports_raw = pd.read_csv('../../data/rawdata/airports.dat.txt', header=None, names=airport_cols)

# Keep only rows with a valid IATA code
coords = (
    airports_raw[airports_raw['iata'].notna() & (airports_raw['iata'] != '\\N')]
    [['iata', 'latitude', 'longitude']]
    .drop_duplicates('iata')
)

print(f"Airports with valid IATA codes: {len(coords):,}")
coords.head()

Airports with valid IATA codes: 6,072


,iata,latitude,longitude
0,GKA,-6.081690,145.391998
1,MAG,-5.207080,145.789001
2,HGU,-5.826790,144.296005
3,LAE,-6.569803,146.725977
4,POM,-9.443380,147.220001


In [8]:
df_geo = (
    df_clean
    .merge(
        coords.rename(columns={'iata': 'ORIGIN', 'latitude': 'ORIGIN_LAT', 'longitude': 'ORIGIN_LON'}),
        on='ORIGIN', how='inner'
    )
    .merge(
        coords.rename(columns={'iata': 'DEST', 'latitude': 'DEST_LAT', 'longitude': 'DEST_LON'}),
        on='DEST', how='inner'
    )
)

print(f"After geo join: {df_geo.shape[0]:,} rows, {df_geo.shape[1]} columns")
print(f"Rows lost in join (airports not in coords): {df_clean.shape[0] - df_geo.shape[0]:,}")
df_geo.head()

After geo join: 32,520,273 rows, 19 columns
Rows lost in join (airports not in coords): 21,575


,YEAR,QUARTER,ORIGIN_AIRPORT_ID,ORIGIN,DEST_AIRPORT_ID,DEST,REPORTING_CARRIER,TICKET_CARRIER,OPERATING_CARRIER,BULK_FARE,PASSENGERS,MARKET_FARE,MARKET_DISTANCE,NONSTOP_MILES,MKT_GEO_TYPE,ORIGIN_LAT,ORIGIN_LON,DEST_LAT,DEST_LON
0,2024,3,10135,ABE,10136,ABI,MQ,AA,99,0.0,1.0,430.5,1575.0,1458.0,2,40.6521,-75.440804,32.411301,-99.681900
1,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,5.5,1961.0,1738.0,2,40.6521,-75.440804,35.040199,-106.609001
2,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,148.5,1961.0,1738.0,2,40.6521,-75.440804,35.040199,-106.609001
3,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,182.0,1961.0,1738.0,2,40.6521,-75.440804,35.040199,-106.609001
4,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,309.5,1961.0,1738.0,2,40.6521,-75.440804,35.040199,-106.609001


In [9]:
print("=== Final Dataset Summary ===")
print(f"Shape: {df_geo.shape}")
print(f"\nMissing values: {df_geo.isna().sum().sum()}")
print(f"\nRow counts by quarter:")
print(df_geo.groupby(['YEAR', 'QUARTER']).size().to_string())
print(f"\nUnique origin airports:  {df_geo['ORIGIN'].nunique():,}")
print(f"Unique dest airports:    {df_geo['DEST'].nunique():,}")
print(f"Unique routes:           {df_geo.groupby(['ORIGIN', 'DEST']).ngroups:,}")
print(f"\nMARKET_FARE stats:")
print(df_geo['MARKET_FARE'].describe())

=== Final Dataset Summary ===
Shape: (32520273, 19)

Missing values: 0

Row counts by quarter:
YEAR  QUARTER
2024  3          8284996
      4          8512287
2025  1          7285614
      2          8437376

Unique origin airports:  451
Unique dest airports:    453
Unique routes:           86,196

MARKET_FARE stats:
count    3.252027e+07
mean     2.710284e+02
std      2.215013e+02
min      1.100000e-01
25%      1.490000e+02
50%      2.309200e+02
75%      3.398700e+02
max      4.443200e+04
Name: MARKET_FARE, dtype: float64


## 6. Save Clean Data

In [10]:
out_path = '../../data/clean_data/T_DB1B_MARKET_CLEAN.csv'
df_geo.to_csv(out_path, index=False)
print(f"Saved {df_geo.shape[0]:,} rows × {df_geo.shape[1]} columns → {out_path}")

Saved 32,520,273 rows × 19 columns → ../../data/clean_data/T_DB1B_MARKET_CLEAN.csv
